# Chapter 4 — Communication Systems & the Link Budget — equations

Standalone, runnable subset of the master `../RF_Equations.ipynb`, scoped to this chapter.
Run top-to-bottom: **Setup**, then this chapter's sections. All functions are verified against the book's worked examples.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C = 2.99792458e8        # speed of light, m/s
EPS0 = 8.8541878128e-12 # vacuum permittivity, F/m

def wavelength(f_hz):
    return C / f_hz


## 1. Free-Space Path Loss (FSPL) — Ch 4.2

$$\mathrm{FSPL(dB)} = 20\log_{10}\!\left(\frac{4\pi d f}{c}\right)
= 32.44 + 20\log_{10} d_{km} + 20\log_{10} f_{MHz}$$

**Engine hook:** the Tier-0 sanity floor. Any `PL(x,y)` cell must be ≥ FSPL at that range.


In [ ]:
def fspl_db(d_m, f_hz):
    # Free-space path loss in dB. d_m > 0.
    return 20*np.log10(4*np.pi*np.asarray(d_m, float)*f_hz / C)

def fspl_db_eng(d_km, f_mhz):
    # Engineering form; equivalent to fspl_db for the same distance/frequency.
    return 32.44 + 20*np.log10(d_km) + 20*np.log10(f_mhz)

# demo: 2.4 GHz over 1..100 m
f = 2.4e9
d = np.linspace(1, 100, 200)
plt.figure(figsize=(6,4))
plt.plot(d, fspl_db(d, f))
plt.xlabel("distance (m)"); plt.ylabel("FSPL (dB)")
plt.title(f"FSPL at {f/1e9:.1f} GHz"); plt.grid(True); plt.show()

# spot check equivalence of the two forms
print(fspl_db(1000, 2.4e9), fspl_db_eng(1.0, 2400.0))


## 10. Link Budget & Noise Floor — Ch 4.1–4.3

The "so what" layer: turn the engine's `PL(x,y)` into a coverage map. Link margin ties
together EIRP (Ch 3), path loss (the engine), Rx gain (Ch 3), and the receiver threshold —
which is set by the noise floor (§4.3). **Coverage = { margin(x,y) > 0 }.** Pipeline Tier 0.

- **Link margin** `M = EIRP − L_path + G_Rx − TH_Rx` (all dB).
- **Friis** `L = G_T G_R (λ/4πd)²` (eq 4.1); FSL without gains = `20log(4πd/λ)` = §1 `fspl_db`.
  The λ-term is only there because Friis uses Rx *gain* instead of effective area `Ae=Gλ²/4π` (Ch 3).
- **Thermal noise** `N = kT₀B` (eq 4.4) → floor `−174 dBm/Hz + 10log B + NF`; `NF = 1 + Te/T₀`.
- **Cascade** `F_tot = F₁ + (F₂−1)/G₁ + (F₃−1)/(G₁G₂) + …` — first stage (LNA) sets the floor;
  passive loss *before* it adds dB-for-dB. *(The book's printed eq 4.14 drops the −1, but its
  Example 4.3 answer only works with the −1 — so this uses the standard, example-consistent form.)*

§4.3 (noise) is receiver-system, not propagation — the engine doesn't compute it, but it sets
`TH_Rx`, the coverage cutoff the map is thresholded against.


In [ ]:
def link_margin_db(eirp_dbm, path_loss_db, g_rx_db, th_rx_dbm):
    # Master link-budget equation (all dB). Margin > 0 => the link closes.
    return eirp_dbm - path_loss_db + g_rx_db - th_rx_dbm

def friis_received_power_dbm(pt_dbm, gt_db, gr_db, d_m, f_hz):
    # Pr = Pt + Gt + Gr - FSPL (dB form of Friis, eq 4.1).
    return pt_dbm + gt_db + gr_db - fspl_db(d_m, f_hz)

# Example 4.1: 100 m, 10 GHz, Pt=0.1 W (20 dBm), Gt=Gr=5 dB, TH=-85 dBm
eirp = 20 + 5                       # dBm (Pt + Gt)
pl   = fspl_db(100, 10e9)
print(f"Ex 4.1: EIRP={eirp} dBm, FSL={pl:.1f} dB, margin={link_margin_db(eirp, pl, 5, -85):.1f} dB (book 22.6 dB)")


In [ ]:
def thermal_noise_dbm(B_hz, nf_db=0.0):
    # AWGN floor: -174 dBm/Hz (kT0 at 290 K) + 10log10(B) + noise figure.
    return -174.0 + 10*np.log10(B_hz) + nf_db

def noise_figure_db_from_temp(Te_K, T0=290.0):
    return 10*np.log10(1 + Te_K/T0)                 # F = 1 + Te/T0 (eq 4.12), in dB

def cascade_noise_factor(F_list, G_list):
    # Friis cascade, linear noise factors F and gains G (standard -1 form; see note above).
    F_tot = F_list[0]; g = 1.0
    for i in range(1, len(F_list)):
        g *= G_list[i-1]
        F_tot += (F_list[i] - 1.0)/g
    return F_tot

# Example 4.2: 10 Msym/s -> B ~ 10 MHz, Te=870 K
nf = noise_figure_db_from_temp(870)
N  = thermal_noise_dbm(10e6, nf)
print(f"Ex 4.2: NF={nf:.1f} dB, N={N:.0f} dBm = {N-30:.0f} dBW (book 6 dB, -128 dBW)")

# Example 4.3: 7 dB cable loss before a receiver of Te=630 K
nf_rx = noise_figure_db_from_temp(630)
print(f"Ex 4.3: Rx NF={nf_rx:.1f} dB + 7 dB cable = {nf_rx+7:.1f} dB (simple add)")
F_cable = 10**(7/10.0); G_cable = 1/F_cable         # passive attenuator: F = L, G = 1/L
F_tot = cascade_noise_factor([F_cable, 10**(nf_rx/10.0)], [G_cable, 1.0])
print(f"        cascade check: {10*np.log10(F_tot):.1f} dB")


In [ ]:
# Synthesis: noise floor -> threshold -> path-loss budget the engine's PL(x,y) is compared to
B, NF, snr_req = 20e6, 6.0, 10.0                    # 20 MHz, 6 dB NF, need 10 dB SNR
th = thermal_noise_dbm(B, NF) + snr_req             # receiver threshold, dBm
print(f"floor {thermal_noise_dbm(B,NF):.1f} dBm + SNR {snr_req:.0f} dB  ->  TH_Rx = {th:.1f} dBm")
eirp, gr = 25.0, 5.0
print(f"budget: EIRP {eirp:.0f} + Gr {gr:.0f} - TH {th:.1f} = {eirp+gr-th:.1f} dB path loss allowed")
print("=> coverage(x,y) = PL(x,y) < this  (equivalently link_margin > 0)")


## 11. Detailed Link Budget, Interference & Eb/N0 — Ch 4.4–4.6

The full itemized budget (book Fig 4.4) + interference margin + the SNR→Eb/N0 step. Fig 4.4 **is**
the engine's per-cell coverage template: at each (x,y), RSL = EIRP − PL(x,y) + Rx_gain, then
net margin = RSL − interference_margin − TH. The engine supplies PL(x,y); everything else is scalar.

- **EIRP** = P_Tx + G_Tx − L_WG − L_radome; **Rx gain** = G_Rx − L_radome − L_WG − L_pol − L_pt.
- **Interference margin** (Ex 4.4): a 1-dB margin ⇒ total interference must stay ≥ 5.9 dB below the
  noise floor. → `interference_for_margin_dbm()`.
- **Eb/N0 = SNR + 10log(B/Rb)** (use the *data* bit rate Rb). → `eb_n0_db()`.
- ⚠ The book's inline 4.5.2–4.5.5 numbers (PL 135 dB, EIRP 27 dBm, SNR 16 dB, 10log B = 60) are
  mutually inconsistent and mismatch Fig 4.4; the self-consistent Fig 4.4 set is used here.


In [ ]:
def interference_for_margin_dbm(noise_dbm, margin_db):
    # Max total interference power (dBm) for a given interference-margin (noise-floor rise).
    return noise_dbm + 10*np.log10(10**(margin_db/10.0) - 1)

def interference_margin_db(noise_dbm, interference_dbm):
    return 10*np.log10(1 + 10**((interference_dbm - noise_dbm)/10.0))

def eb_n0_db(snr_db, B_hz, Rb_bps):
    # Eb/N0 (dB) = SNR + 10log10(B/Rb). Use the DATA bit rate Rb, not the channel rate.
    return snr_db + 10*np.log10(B_hz/Rb_bps)

# Example 4.4: a 1-dB interference margin
print(f"Ex 4.4: 1-dB margin -> interference must stay {-interference_for_margin_dbm(0,1.0):.1f} dB below noise (book 5.9)")


In [ ]:
def link_budget(f_hz, d_m, tx_pwr_dbm, tx_gain_db, tx_loss_db, tx_radome_db,
                pl_extra_db, rx_gain_db, rx_radome_db, rx_loss_db, rx_pol_db, rx_pt_db,
                nf_db, bw_hz, snr_req_db, interf_margin_db=0.0):
    eirp = tx_pwr_dbm + tx_gain_db - tx_loss_db - tx_radome_db
    fsl = fspl_db(d_m, f_hz)
    total_pl = fsl + pl_extra_db
    rx_gain = rx_gain_db - rx_radome_db - rx_loss_db - rx_pol_db - rx_pt_db
    rsl = eirp - total_pl + rx_gain
    noise = thermal_noise_dbm(bw_hz, nf_db)
    snr = rsl - noise - interf_margin_db
    threshold = noise + snr_req_db
    net_margin = rsl - interf_margin_db - threshold
    return dict(EIRP=eirp, FSL=fsl, total_PL=total_pl, Rx_gain=rx_gain, RSL=rsl,
                noise=noise, SNR=snr, threshold=threshold, net_margin=net_margin)

# Reproduce book Figure 4.4: 38.6 GHz, 2 km terrestrial mmwave link
lb = link_budget(38.6e9, 2000, tx_pwr_dbm=10, tx_gain_db=32, tx_loss_db=1.5, tx_radome_db=2.0,
                 pl_extra_db=1.0+15.0+2.0+0.2,     # pointing + rain(0.999) + multipath + atmos
                 rx_gain_db=32, rx_radome_db=2.0, rx_loss_db=2.0, rx_pol_db=0.2, rx_pt_db=1.0,
                 nf_db=7.0, bw_hz=25e6, snr_req_db=5.0, interf_margin_db=1.0)
for k in ("EIRP","FSL","total_PL","Rx_gain","RSL","noise","SNR","threshold","net_margin"):
    print(f"  {k:11s} {lb[k]:7.1f}")
print("book Fig 4.4:  EIRP 38.5  FSL 130.2  PL 148.4  RxG 26.8  RSL -83.1  N -93.0  SNR 8.9  TH -88.0  margin 3.9")
